# 04 · Trained sentiment model (MLflow + Unity Catalog)
Calling an LLM for sentiment on every sentence, every day, is a recurring token bill. Instead:
1. **Weak-label** a sample of distinct customer sentences once with `ai_analyze_sentiment`.
2. **Train** a small TF-IDF + logistic-regression model on those labels.
3. **Track + register** it in MLflow / Unity Catalog (`@champion` alias) and score every sentence at ~zero cost.

Trade-off (documented): weak labels are not human ground truth. The next step with a real
customer is to have analysts adjudicate a few hundred sentences and retrain.

In [ ]:
%pip install -q scikit-learn==1.5.2
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
import json, os
import mlflow, numpy as np, pandas as pd
from mlflow.models import infer_signature
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

for k, v in {"catalog": "daria_krasavina", "schema": "voc_valtera", "label_sample": "2500",
             "evidence_path": "/Volumes/daria_krasavina/voc_valtera/evidence"}.items():
    dbutils.widgets.text(k, v)
cat, sch, ev = (dbutils.widgets.get(k) for k in ["catalog", "schema", "evidence_path"])
n_label = int(dbutils.widgets.get("label_sample"))
ns = f"{cat}.{sch}"
MODEL_NAME = f"{ns}.voc_sentiment_model"

In [ ]:
# 1. Weak labels (cached: only new sentences are sent to the LLM)
spark.sql(f"""CREATE TABLE IF NOT EXISTS {ns}.sentiment_weak_labels
  (sentence_text STRING, weak_label STRING, labeled_at TIMESTAMP)
  COMMENT 'ai_analyze_sentiment weak labels used to train voc_sentiment_model (not human ground truth).'""")
spark.sql(f"""
  INSERT INTO {ns}.sentiment_weak_labels
  SELECT sentence_text, ai_analyze_sentiment(sentence_text), current_timestamp()
  FROM (SELECT DISTINCT sentence_text FROM {ns}.silver_sentences
        WHERE speaker = 'customer' AND size(split(sentence_text, ' ')) >= 3
        EXCEPT SELECT sentence_text FROM {ns}.sentiment_weak_labels
        ORDER BY rand(11) LIMIT {n_label})""")
lab = spark.sql(f"""SELECT sentence_text,
                   CASE WHEN weak_label = 'mixed' THEN 'negative' ELSE weak_label END AS label
                   FROM {ns}.sentiment_weak_labels WHERE weak_label IS NOT NULL""").toPandas()
print(lab.label.value_counts().to_string())

{"ts": "2026-11-20 19:42:05.729", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Cilqb2JzLzc5MTg0NTUxMDUwNzk3Mi9ydW5zLzU0NDUyNTE4MTEzNTQ5NRABIAEqJDk1NzdjY2UyLWYzZGQtNGI0Zi1hNjU3LWIxNjA3ZTM4MDFkOTIkMDFhMWZhN2EtNmM0OC03YmEyLTgzNDgtYjk5MjIzY2YxNWM5OiRlMGUzYzc1ZS1hMWEwLTNlMWItYTQ0MS1iOWZhNTNlZjVjYTFCJDk1NzdjY2UyLWYzZGQtNGI0Zi1hNjU3LWIxNjA3ZTM4MDFkOUoMCOnEgtgGEICQs4MCUAJYAWABaJfDg9WK9N8BiAEA.", "context": {}}
{"ts": "2026-11-20 19:42:05.729", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Cilqb2JzLzc5MTg0NTUxMDUwNzk3Mi9ydW5zLzU0NDUyNTE4MTEzNTQ5NRABIAEqJDk1NzdjY2UyLWYzZGQtNGI0Zi1hNjU3LWIxNjA3ZTM4MDFkOTIkMDFhMWZhN2EtNmM0OC03YmEyLTgzNDgtYjk5MjIzY2YxNWM5OiRlMGUzYzc1ZS1hMWEwLTNlMWItYTQ0MS1iOWZhNTNlZjVjYTFCJDk1NzdjY2UyLWYzZGQtNGI0Zi1hNjU3LWIxNjA3ZTM4MDFkOUoMCOnEgtgGEICQs4MCUAJYAWABaJfDg9WK9N8BiAEA.", "context": {}}
{"ts": "2026-11-20 19:42:05.729", "level": "WA

In [ ]:
# 2. Train + 3. log/register
X_tr, X_te, y_tr, y_te = train_test_split(lab.sentence_text, lab.label, test_size=0.2,
                                          random_state=7, stratify=lab.label)
pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
                 ("clf", LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced"))])
me = spark.sql("SELECT current_user()").first()[0]
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{me}/valtera-voc-sentiment")
with mlflow.start_run(run_name="tfidf_logreg_weak_labels") as run:
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    metrics = {"accuracy": accuracy_score(y_te, pred), "macro_f1": f1_score(y_te, pred, average="macro"),
               "n_train": len(X_tr), "n_test": len(X_te)}
    mlflow.log_params({"vectorizer": "tfidf_1_2gram", "model": "logreg_C4_balanced", "label_source": "ai_analyze_sentiment"})
    mlflow.log_metrics(metrics)
    report = classification_report(y_te, pred)
    mlflow.log_text(report, "classification_report.txt")
    info = mlflow.sklearn.log_model(pipe, name="model", registered_model_name=MODEL_NAME,
                                    signature=infer_signature(X_tr.to_frame(), pred),
                                    input_example=X_tr.head(3).to_frame())
print(json.dumps(metrics, indent=2)); print(report)
from mlflow import MlflowClient
client = MlflowClient()
version = info.registered_model_version
client.set_registered_model_alias(MODEL_NAME, "champion", version)
print(f"registered {MODEL_NAME} v{version} @champion")

🔗 View Logged Model at: https://adb-984752964297111.11.azuredatabricks.net/ml/experiments/2686806295104837/models/m-cdd05a3f76a04cbd952a71308eae0185?o=984752964297111
Registered model 'daria_krasavina.voc_valtera.voc_sentiment_model' already exists. Creating a new version of this model...
{'text/plain': 'Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': 'dec4289df0b3421d8fd9900d4493a144'}}🔗 Created version '10' of model 'daria_krasavina.voc_valtera.voc_sentiment_model': https://adb-984752964297111.11.azuredatabricks.net/explore/data/models/daria_krasavina/voc_valtera/voc_sentiment_model/version/10?o=984752964297111
{
  "accuracy": 0.9873417721518988,
  "macro_f1": 0.9819804338032377,
  "n_train": 3158,
  "n_test": 790
}
              precision    recall  f1-score   support

    negative       0.99      0.99      0.99       580
     neutral       0.97      0.97      0.97       14

In [ ]:
# Score every customer sentence with the @champion model (distinct texts, joined back)
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
texts = spark.sql(f"SELECT DISTINCT sentence_text FROM {ns}.silver_sentences WHERE speaker='customer'").toPandas()
proba = champion.predict_proba(texts.sentence_text)
classes = list(champion.classes_)
texts["sentiment_label"] = np.array(classes)[proba.argmax(1)]
texts["negative_prob"] = proba[:, classes.index("negative")]
texts["sentiment_score"] = proba[:, classes.index("positive")] - texts["negative_prob"]
spark.createDataFrame(texts).createOrReplaceTempView("scored_texts")
spark.sql(f"""CREATE OR REPLACE TABLE {ns}.sentence_sentiment
  COMMENT 'Sentence sentiment from voc_sentiment_model@champion (score = P(positive) - P(negative)).' AS
  SELECT s.sentence_id, s.interaction_id, t.sentiment_label, t.negative_prob, t.sentiment_score,
         '{MODEL_NAME}' AS model_name, {version} AS model_version
  FROM {ns}.silver_sentences s JOIN scored_texts t USING (sentence_text)
  WHERE s.speaker = 'customer'""")
dist = spark.sql(f"SELECT sentiment_label, count(*) n FROM {ns}.sentence_sentiment GROUP BY 1").toPandas()
print(dist.to_string(index=False))
os.makedirs(ev, exist_ok=True)
with open(f"{ev}/04_sentiment_model.json", "w") as f:
    json.dump({"model": MODEL_NAME, "version": int(version), "alias": "champion", "mlflow_run_id": run.info.run_id,
               "metrics_vs_weak_labels": metrics, "classification_report": report,
               "weak_label_distribution": lab.label.value_counts().to_dict(),
               "scored_distribution": dist.to_dict("records")}, f, indent=2)

{'text/plain': 'Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '27e4d46324c44850b01a4f446c311891'}}sentiment_label     n
       positive  3532
       negative 11983
        neutral  7230
